In [1]:
!git clone https://github.com/Colin-0625/xiaohongshu-multimodal-ablation.git
%cd xiaohongshu-multimodal-ablation


Cloning into 'xiaohongshu-multimodal-ablation'...
remote: Enumerating objects: 49, done.
remote: Enumerating objects: 49, done.
remote: Total 49 (delta 0), reused 0 (delta 0), pack-reused 49 (from 1)
Receiving objects: 100% (49/49), 42.29 MiB | 18.75 MiB/s, done.
Resolving deltas: 100% (20/20), done.
remote: Total 49 (delta 0), reused 0 (delta 0), pack-reused 49 (from 1)
Receiving objects: 100% (49/49), 42.29 MiB | 18.75 MiB/s, done.
Resolving deltas: 100% (20/20), done.
/content/xiaohongshu-multimodal-ablation
/content/xiaohongshu-multimodal-ablation


In [2]:
!pip install -r requirements.txt -q

In [4]:
!unzip -q data/images.zip -d data/

In [5]:
!ls data/images/

part_113  part_114  part_115  part_116


In [6]:
!mkdir -p features results results/figures

In [7]:
!python src/extract_text_features.py

Loaded 714 samples from data/metadata_balanced.csv
label
knowledge_tutorial    238
fashion_beauty        238
food_travel           238
Name: count, dtype: int64
Loading tokenizer and model: hfl/chinese-roberta-wwm-ext
tokenizer_config.json: 100% 19.0/19.0 [00:00<00:00, 58.6kB/s]
vocab.txt: 110kB [00:00, 27.5MB/s]
tokenizer.json: 269kB [00:00, 20.9MB/s]
added_tokens.json: 100% 2.00/2.00 [00:00<00:00, 10.5kB/s]
special_tokens_map.json: 100% 112/112 [00:00<00:00, 452kB/s]
config.json: 100% 689/689 [00:00<00:00, 2.86MB/s]
pytorch_model.bin: 100% 412M/412M [00:03<00:00, 105MB/s]
Loading weights:  11% 22/199 [00:00<00:00, 3317.44it/s, Materializing param=encoder.layer.1.attention.output.LayerNorm.bias]Exception in thread Thread-auto_conversion:
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_http.py", line 761, in hf_raise_for_status
    response.raise_for_status()
  File "/usr/local/lib/python3.12/dist-packages/httpx/_models.py", lin

In [8]:
!python src/extract_image_features.py

Loaded metadata: 714 samples
Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth
100% 97.8M/97.8M [00:00<00:00, 191MB/s]
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
Extracting image features:   0% 0/12 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create

In [9]:
# 重新生成一个数字版的 labels
import numpy as np

labels = np.load('features/text_labels.npy', allow_pickle=True)
ids    = np.load('features/text_ids.npy',    allow_pickle=True)

label2idx = {'fashion_beauty': 0, 'food_travel': 1, 'knowledge_tutorial': 2}
labels_int = np.array([label2idx[l] for l in labels], dtype=np.int64)

np.save('features/text_labels.npy',  labels_int)
np.save('features/image_labels.npy', labels_int)  # image那个也一起修复

print("Done:", labels_int.shape, labels_int[:5])

Done: (714,) [2 0 2 0 1]


In [10]:
!python src/train_text_only.py

Train: 570  Val: 72  Test: 72
Device: cuda
Epoch   1/50 | loss=0.8671 | val_acc=0.7083 | val_f1=0.7148
Epoch   5/50 | loss=0.3423 | val_acc=0.7222 | val_f1=0.7202
Epoch  10/50 | loss=0.1925 | val_acc=0.7222 | val_f1=0.7211
Epoch  15/50 | loss=0.1108 | val_acc=0.7361 | val_f1=0.7362
Epoch  20/50 | loss=0.0816 | val_acc=0.7222 | val_f1=0.7236
Epoch  25/50 | loss=0.0563 | val_acc=0.7361 | val_f1=0.7362
Epoch  30/50 | loss=0.0399 | val_acc=0.7222 | val_f1=0.7221
Epoch  35/50 | loss=0.0323 | val_acc=0.7361 | val_f1=0.7358
Epoch  40/50 | loss=0.0291 | val_acc=0.7361 | val_f1=0.7358
Epoch  45/50 | loss=0.0251 | val_acc=0.7361 | val_f1=0.7358
Epoch  50/50 | loss=0.0231 | val_acc=0.7361 | val_f1=0.7358

===== Text-Only Results =====
Test Accuracy : 0.8056
Test Macro-F1 : 0.8022
Best model saved to: results/text_only_best.pt


In [11]:
!python src/train_image_only.py

Train: 570  Val: 72  Test: 72
Device: cuda
Epoch   1/50 | loss=0.9767 | val_acc=0.6389 | val_f1=0.6380
Epoch   5/50 | loss=0.3557 | val_acc=0.6528 | val_f1=0.6511
Epoch  10/50 | loss=0.1294 | val_acc=0.6389 | val_f1=0.6341
Epoch  15/50 | loss=0.0593 | val_acc=0.6250 | val_f1=0.6189
Epoch  20/50 | loss=0.0414 | val_acc=0.5833 | val_f1=0.5724
Epoch  25/50 | loss=0.0301 | val_acc=0.5972 | val_f1=0.5892
Epoch  30/50 | loss=0.0259 | val_acc=0.5972 | val_f1=0.5892
Epoch  35/50 | loss=0.0211 | val_acc=0.5972 | val_f1=0.5892
Epoch  40/50 | loss=0.0188 | val_acc=0.5972 | val_f1=0.5897
Epoch  45/50 | loss=0.0166 | val_acc=0.5833 | val_f1=0.5770
Epoch  50/50 | loss=0.0153 | val_acc=0.5694 | val_f1=0.5632

===== Image-Only Results =====
Test Accuracy : 0.6250
Test Macro-F1 : 0.6238
Best model saved to: results/image_only_best.pt


In [12]:
!python src/train_late_fusion.py

Train: 570  Val: 72  Test: 72
Device: cuda
Epoch   1/50 | loss=0.8106 | val_acc=0.7083 | val_f1=0.7033
Epoch   5/50 | loss=0.1815 | val_acc=0.7361 | val_f1=0.7351
Epoch  10/50 | loss=0.0293 | val_acc=0.7083 | val_f1=0.7074
Epoch  15/50 | loss=0.0069 | val_acc=0.7083 | val_f1=0.7073
Epoch  20/50 | loss=0.0055 | val_acc=0.6944 | val_f1=0.6909
Epoch  25/50 | loss=0.0036 | val_acc=0.6667 | val_f1=0.6643
Epoch  30/50 | loss=0.0030 | val_acc=0.6806 | val_f1=0.6782
Epoch  35/50 | loss=0.0027 | val_acc=0.6806 | val_f1=0.6782
Epoch  40/50 | loss=0.0025 | val_acc=0.6667 | val_f1=0.6643
Epoch  45/50 | loss=0.0022 | val_acc=0.6667 | val_f1=0.6643
Epoch  50/50 | loss=0.0023 | val_acc=0.6667 | val_f1=0.6643

===== Late Fusion Results =====
Test Accuracy : 0.8333
Test Macro-F1 : 0.8344
Best model saved to: results/late_fusion_best.pt


In [13]:
!python src/train_gated_fusion.py

Max content length in dataset: 1187
Train: 570  Val: 72  Test: 72
Device: cuda
Epoch   1/50 | loss=0.8001 | val_acc=0.7083 | val_f1=0.6961
Epoch   5/50 | loss=0.2001 | val_acc=0.7361 | val_f1=0.7368
Epoch  10/50 | loss=0.0253 | val_acc=0.7083 | val_f1=0.7072
Epoch  15/50 | loss=0.0053 | val_acc=0.6806 | val_f1=0.6811
Epoch  20/50 | loss=0.0037 | val_acc=0.6806 | val_f1=0.6811
Epoch  25/50 | loss=0.0029 | val_acc=0.6806 | val_f1=0.6811
Epoch  30/50 | loss=0.0024 | val_acc=0.6806 | val_f1=0.6811
Epoch  35/50 | loss=0.0021 | val_acc=0.6806 | val_f1=0.6811
Epoch  40/50 | loss=0.0019 | val_acc=0.6806 | val_f1=0.6811
Epoch  45/50 | loss=0.0018 | val_acc=0.6806 | val_f1=0.6811
Epoch  50/50 | loss=0.0017 | val_acc=0.6806 | val_f1=0.6811

===== Gated Fusion Results =====
Test Accuracy : 0.8056
Test Macro-F1 : 0.8074
Best model saved to: results/gated_fusion_best.pt
Gate weights saved to: results/gate_weights_test.npy

Average gate weight (image reliance) per class:
  fashion_beauty: 0.3755
  fo

In [14]:
!python src/evaluate.py

Loading features …
Test samples: 72
Loading model weights …
Running inference …

── text-only ──
                    precision    recall  f1-score   support

    fashion_beauty       0.89      0.67      0.76        24
       food_travel       0.82      0.96      0.88        24
knowledge_tutorial       0.73      0.79      0.76        24

          accuracy                           0.81        72
         macro avg       0.81      0.81      0.80        72
      weighted avg       0.81      0.81      0.80        72


── image-only ──
                    precision    recall  f1-score   support

    fashion_beauty       0.72      0.54      0.62        24
       food_travel       0.72      0.54      0.62        24
knowledge_tutorial       0.53      0.79      0.63        24

          accuracy                           0.62        72
         macro avg       0.66      0.62      0.62        72
      weighted avg       0.66      0.62      0.62        72


── late fusion ──
                    

---
## Part 2 — ResNet18 vs ResNet50 Ablation

### Step A1: Extract ResNet18 features

In [15]:
import os, numpy as np, pandas as pd
from PIL import Image
import torch
from torchvision.models import resnet18, ResNet18_Weights
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

METADATA  = "data/metadata_balanced.csv"
IMG_ROOT  = "data/images"
FEAT_DIR  = "features"
BATCH     = 64
DEVICE    = "cuda" if torch.cuda.is_available() else "cpu"
LABEL2IDX = {"fashion_beauty":0,"food_travel":1,"knowledge_tutorial":2}

weights  = ResNet18_Weights.DEFAULT
model18  = resnet18(weights=weights)
model18.fc = torch.nn.Identity()   # output: 512-dim
model18  = model18.to(DEVICE).eval()
transform = weights.transforms()

df = pd.read_csv(METADATA)

class ImgDS(torch.utils.data.Dataset):
    def __init__(self, df, tfm):
        self.df, self.tfm = df.reset_index(drop=True), tfm
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        rel = row["image_path"].replace("image/","",1)
        path = os.path.join(IMG_ROOT, rel)
        try:
            img = Image.open(path).convert("RGB")
            img = self.tfm(img)
        except:
            img = torch.zeros(3,224,224)
        return img, LABEL2IDX[row["label"]], row["note_idx"]

loader = DataLoader(ImgDS(df, transform), batch_size=BATCH,
                    shuffle=False, num_workers=2)

feats, labs, ids = [], [], []
with torch.no_grad():
    for imgs, l, nid in tqdm(loader, desc="ResNet18 features"):
        feats.append(model18(imgs.to(DEVICE)).cpu().numpy())
        labs.extend(l.numpy()); ids.extend(nid.numpy().tolist())

feats18 = np.vstack(feats).astype(np.float32)
np.save(f"{FEAT_DIR}/image_features_r18.npy", feats18)
np.save(f"{FEAT_DIR}/image_ids_r18.npy", np.array(ids))
print(f"ResNet18 features saved: {feats18.shape}")


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 193MB/s]
ResNet18 features: 100%|██████████| 12/12 [00:38<00:00,  3.17s/it]

ResNet18 features saved: (714, 512)


### Step A2: Train image-only and fusion models with ResNet18

In [16]:
import torch, torch.nn as nn, numpy as np, pandas as pd
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import accuracy_score, f1_score

SPLITS = "data/splits"; RESULTS = "results"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
EPOCHS=50; BS=64; LR=1e-3; WD=1e-4; SEED=42
torch.manual_seed(SEED); np.random.seed(SEED)

def get_mask(ids, csv):
    split_ids = set(pd.read_csv(csv)["note_idx"].tolist())
    return np.array([i in split_ids for i in ids])

def make_loader(tensors, shuffle=True):
    return DataLoader(TensorDataset(*tensors), batch_size=BS, shuffle=shuffle)

@torch.no_grad()
def evaluate(model, loader, fusion=False):
    model.eval(); preds, labels = [], []
    for batch in loader:
        if fusion:
            xt, xi, y = batch
            logits = model(xt.to(DEVICE), xi.to(DEVICE))
        else:
            x, y = batch
            logits = model(x.to(DEVICE))
        preds.extend(logits.argmax(1).cpu().numpy())
        labels.extend(y.numpy())
    return accuracy_score(labels,preds), f1_score(labels,preds,average="macro")

def train_model(model, train_loader, val_loader, save_path, fusion=False):
    opt = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WD)
    sch = torch.optim.lr_scheduler.StepLR(opt, step_size=15, gamma=0.5)
    crit = nn.CrossEntropyLoss(); best_f1=0
    for ep in range(1, EPOCHS+1):
        model.train()
        for batch in train_loader:
            if fusion:
                xt, xi, y = batch
                loss = crit(model(xt.to(DEVICE), xi.to(DEVICE)), y.to(DEVICE))
            else:
                x, y = batch
                loss = crit(model(x.to(DEVICE)), y.to(DEVICE))
            opt.zero_grad(); loss.backward(); opt.step()
        sch.step()
        _, vf1 = evaluate(model, val_loader, fusion)
        if vf1 > best_f1:
            best_f1=vf1; torch.save(model.state_dict(), save_path)
    model.load_state_dict(torch.load(save_path, map_location=DEVICE))
    return model

# ── Load features ──────────────────────────────────────
txt_feat = np.load("features/text_features.npy")
txt_lbl  = np.load("features/text_labels.npy")
txt_ids  = np.load("features/text_ids.npy")
img18    = np.load("features/image_features_r18.npy")
ids18    = np.load("features/image_ids_r18.npy")
assert np.array_equal(txt_ids, ids18), "ID mismatch!"

for split in ["train","val","test"]:
    mask = get_mask(txt_ids, f"{SPLITS}/{split}.csv")
    globals()[f"Xt_{split}"] = torch.tensor(txt_feat[mask], dtype=torch.float32)
    globals()[f"Xi18_{split}"] = torch.tensor(img18[mask], dtype=torch.float32)
    globals()[f"y_{split}"] = torch.tensor(txt_lbl[mask], dtype=torch.long)

# ── Image-only R18 ──────────────────────────────────────
class MLP(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d,256),nn.ReLU(),nn.Dropout(0.2),nn.Linear(256,3))
    def forward(self, x): return self.net(x)

class LateFusion(nn.Module):
    def __init__(self, td, id_):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(td+id_,512),nn.ReLU(),nn.Dropout(0.3),nn.Linear(512,3))
    def forward(self, t, v): return self.net(torch.cat([t,v],dim=1))

print("Training image-only R18...")
m_img18 = MLP(512).to(DEVICE)
tr = make_loader((Xi18_train, y_train))
va = make_loader((Xi18_val, y_val), False)
te = make_loader((Xi18_test, y_test), False)
train_model(m_img18, tr, va, f"{RESULTS}/image_only_r18_best.pt")
acc, f1 = evaluate(m_img18, te)
print(f"Image-only R18  — Acc: {acc:.4f}  Macro-F1: {f1:.4f}")
r18_img_acc, r18_img_f1 = acc, f1

print("\nTraining late fusion R18...")
m_lf18 = LateFusion(768, 512).to(DEVICE)
tr = make_loader((Xt_train, Xi18_train, y_train))
va = make_loader((Xt_val,   Xi18_val,   y_val),   False)
te = make_loader((Xt_test,  Xi18_test,  y_test),  False)
train_model(m_lf18, tr, va, f"{RESULTS}/late_fusion_r18_best.pt", fusion=True)
acc, f1 = evaluate(m_lf18, te, fusion=True)
print(f"Late Fusion  R18 — Acc: {acc:.4f}  Macro-F1: {f1:.4f}")
r18_lf_acc, r18_lf_f1 = acc, f1


Training image-only R18...
Image-only R18  — Acc: 0.5833  Macro-F1: 0.5807

Training late fusion R18...
Late Fusion  R18 — Acc: 0.7917  Macro-F1: 0.7849


### Step A3: ResNet18 vs ResNet50 comparison table

In [17]:
print("=" * 60)
print(f"{'Model':<25} {'Accuracy':>10} {'Macro-F1':>10}")
print("-" * 60)
# R50 results from earlier run
results = [
    ("Image-only  (ResNet50)", 0.6250, 0.6238),
    ("Image-only  (ResNet18)", r18_img_acc, r18_img_f1),
    ("Late Fusion (ResNet50)", 0.8333, 0.8344),
    ("Late Fusion (ResNet18)", r18_lf_acc, r18_lf_f1),
    ("Text-only   (baseline)", 0.8056, 0.8022),
]
for name, acc, f1 in results:
    print(f"{name:<25} {acc:>10.4f} {f1:>10.4f}")
print("=" * 60)


Model                       Accuracy   Macro-F1
------------------------------------------------------------
Image-only  (ResNet50)        0.6250     0.6238
Image-only  (ResNet18)        0.5833     0.5807
Late Fusion (ResNet50)        0.8333     0.8344
Late Fusion (ResNet18)        0.7917     0.7849
Text-only   (baseline)        0.8056     0.8022


---
## Part 3 — Multi-Seed Evaluation & Statistical Significance

### Step B1: Run all models with 5 random seeds

In [18]:
import torch, torch.nn as nn, numpy as np, pandas as pd
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import f1_score, accuracy_score
from scipy import stats

SPLITS = "data/splits"; RESULTS = "results"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
EPOCHS=50; BS=64; LR=1e-3; WD=1e-4
SEEDS = [42, 123, 456, 789, 2024]

# Load all features once
txt_feat = np.load("features/text_features.npy")
txt_lbl  = np.load("features/text_labels.npy")
txt_ids  = np.load("features/text_ids.npy")
img50    = np.load("features/image_features.npy")
img18    = np.load("features/image_features_r18.npy")

def get_split(ids, lbl, feat_list, csv):
    split_ids = set(pd.read_csv(csv)["note_idx"].tolist())
    mask = np.array([i in split_ids for i in ids])
    tensors = [torch.tensor(f[mask], dtype=torch.float32) for f in feat_list]
    tensors.append(torch.tensor(lbl[mask], dtype=torch.long))
    return tensors

def make_loader(tensors, shuffle=True):
    return DataLoader(TensorDataset(*tensors), batch_size=BS, shuffle=shuffle)

class TextMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(768,256),nn.ReLU(),nn.Dropout(0.2),nn.Linear(256,3))
    def forward(self,x): return self.net(x)

class ImgMLP(nn.Module):
    def __init__(self, d=2048):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d,256),nn.ReLU(),nn.Dropout(0.2),nn.Linear(256,3))
    def forward(self,x): return self.net(x)

class LateFusion(nn.Module):
    def __init__(self, id_=2048):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(768+id_,512),nn.ReLU(),nn.Dropout(0.3),nn.Linear(512,3))
    def forward(self,t,v): return self.net(torch.cat([t,v],dim=1))

class GatedFusion(nn.Module):
    def __init__(self):
        super().__init__()
        self.tp = nn.Linear(768,256); self.vp = nn.Linear(2048,256)
        self.gate = nn.Sequential(nn.Linear(513,128),nn.ReLU(),nn.Linear(128,1),nn.Sigmoid())
        self.clf  = nn.Linear(256,3)
    def forward(self,t,v,l):
        tp=self.tp(t); vp=self.vp(v)
        g=self.gate(torch.cat([tp,vp,l],dim=1))
        return self.clf(g*vp+(1-g)*tp), g

@torch.no_grad()
def eval_model(model, loader, mode):
    model.eval(); preds, labels = [], []
    for batch in loader:
        if mode=="text":    logits=model(batch[0].to(DEVICE)); y=batch[1]
        elif mode=="img":   logits=model(batch[0].to(DEVICE)); y=batch[1]
        elif mode=="late":  logits=model(batch[0].to(DEVICE),batch[1].to(DEVICE)); y=batch[2]
        elif mode=="gated": logits,_=model(batch[0].to(DEVICE),batch[1].to(DEVICE),batch[2].to(DEVICE)); y=batch[3]
        preds.extend(logits.argmax(1).cpu().numpy()); labels.extend(y.numpy())
    return f1_score(labels,preds,average="macro")

def run_seed(seed):
    torch.manual_seed(seed); np.random.seed(seed)
    crit = nn.CrossEntropyLoss()

    # Get metadata for text lengths (gated fusion)
    meta = pd.read_csv("data/metadata_balanced.csv")
    max_len = meta["content_length"].max()
    id2len  = dict(zip(meta["note_idx"], meta["content_length"]))

    results = {}

    for split in ["train","val","test"]:
        csv = f"{SPLITS}/{split}.csv"
        split_ids = set(pd.read_csv(csv)["note_idx"].tolist())
        mask = np.array([i in split_ids for i in txt_ids])
        globals()[f"Xt_{split}"] = torch.tensor(txt_feat[mask], dtype=torch.float32)
        globals()[f"Xi50_{split}"] = torch.tensor(img50[mask], dtype=torch.float32)
        globals()[f"Xi18_{split}"] = torch.tensor(img18[mask], dtype=torch.float32)
        globals()[f"y_{split}"] = torch.tensor(txt_lbl[mask], dtype=torch.long)
        lens = np.array([id2len[i] for i in txt_ids[mask]], dtype=np.float32) / max_len
        globals()[f"Xl_{split}"] = torch.tensor(lens, dtype=torch.float32).unsqueeze(1)

    def train(model, loader_tr, loader_va, mode):
        opt = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WD)
        sch = torch.optim.lr_scheduler.StepLR(opt, step_size=15, gamma=0.5)
        best=0; best_sd=None
        for _ in range(EPOCHS):
            model.train()
            for batch in loader_tr:
                if mode=="text":   loss=crit(model(batch[0].to(DEVICE)),batch[1].to(DEVICE))
                elif mode=="img":  loss=crit(model(batch[0].to(DEVICE)),batch[1].to(DEVICE))
                elif mode=="late": loss=crit(model(batch[0].to(DEVICE),batch[1].to(DEVICE)),batch[2].to(DEVICE))
                elif mode=="gated":
                    logits,_=model(batch[0].to(DEVICE),batch[1].to(DEVICE),batch[2].to(DEVICE))
                    loss=crit(logits,batch[3].to(DEVICE))
                opt.zero_grad(); loss.backward(); opt.step()
            sch.step()
            vf1 = eval_model(model, loader_va, mode)
            if vf1>best: best=vf1; best_sd={k:v.clone() for k,v in model.state_dict().items()}
        model.load_state_dict(best_sd)
        return model

    # text-only
    m = train(TextMLP().to(DEVICE),
              make_loader((Xt_train,y_train)),
              make_loader((Xt_val,y_val),False), "text")
    results["text_only"] = eval_model(m, make_loader((Xt_test,y_test),False), "text")

    # image-only R50
    m = train(ImgMLP(2048).to(DEVICE),
              make_loader((Xi50_train,y_train)),
              make_loader((Xi50_val,y_val),False), "img")
    results["image_only_r50"] = eval_model(m, make_loader((Xi50_test,y_test),False), "img")

    # image-only R18
    m = train(ImgMLP(512).to(DEVICE),
              make_loader((Xi18_train,y_train)),
              make_loader((Xi18_val,y_val),False), "img")
    results["image_only_r18"] = eval_model(m, make_loader((Xi18_test,y_test),False), "img")

    # late fusion R50
    m = train(LateFusion(2048).to(DEVICE),
              make_loader((Xt_train,Xi50_train,y_train)),
              make_loader((Xt_val,Xi50_val,y_val),False), "late")
    results["late_fusion_r50"] = eval_model(m, make_loader((Xt_test,Xi50_test,y_test),False), "late")

    # late fusion R18
    m = train(LateFusion(512).to(DEVICE),
              make_loader((Xt_train,Xi18_train,y_train)),
              make_loader((Xt_val,Xi18_val,y_val),False), "late")
    results["late_fusion_r18"] = eval_model(m, make_loader((Xt_test,Xi18_test,y_test),False), "late")

    # gated fusion
    m = train(GatedFusion().to(DEVICE),
              make_loader((Xt_train,Xi50_train,Xl_train,y_train)),
              make_loader((Xt_val,Xi50_val,Xl_val,y_val),False), "gated")
    results["gated_fusion"] = eval_model(m, make_loader((Xt_test,Xi50_test,Xl_test,y_test),False), "gated")

    return results

# Run all seeds (takes ~20-30 min on T4)
all_results = {s: run_seed(s) for s in SEEDS}
np.save(f"{RESULTS}/multiseed_results.npy", all_results)
print("Done. Results per seed:")
for s, r in all_results.items():
    print(f"  Seed {s}: {r}")


Done. Results per seed:
  Seed 42: {'text_only': 0.8021733821733822, 'image_only_r50': 0.6226190476190476, 'image_only_r18': 0.664080417703606, 'late_fusion_r50': 0.7878317695593443, 'late_fusion_r18': 0.8451178451178452, 'gated_fusion': 0.8467908902691512}
  Seed 123: {'text_only': 0.8324475642862758, 'image_only_r50': 0.6270748550992723, 'image_only_r18': 0.6521985815602837, 'late_fusion_r50': 0.8477444631876802, 'late_fusion_r18': 0.8444444444444444, 'gated_fusion': 0.7787878787878788}
  Seed 456: {'text_only': 0.8331854480922803, 'image_only_r50': 0.6239878542510121, 'image_only_r18': 0.6240340477276182, 'late_fusion_r50': 0.8344481605351172, 'late_fusion_r18': 0.8307616586686354, 'gated_fusion': 0.8320261437908497}
  Seed 789: {'text_only': 0.8002784739626846, 'image_only_r50': 0.6366433566433567, 'image_only_r18': 0.6098765432098765, 'late_fusion_r50': 0.7899159663865546, 'late_fusion_r18': 0.8192512077294686, 'gated_fusion': 0.8175757575757577}
  Seed 2024: {'text_only': 0.79125

### Step B2: Summary table — mean ± std + t-test vs text-only

In [19]:
import numpy as np
from scipy import stats

all_results = np.load("results/multiseed_results.npy", allow_pickle=True).item()
models = ["text_only","image_only_r50","image_only_r18",
          "late_fusion_r50","late_fusion_r18","gated_fusion"]

scores = {m: [all_results[s][m] for s in all_results] for m in models}
text_scores = scores["text_only"]

print("=" * 75)
print(f"{'Model':<22} {'Mean F1':>9} {'Std':>7} {'vs text-only':>14} {'p-value':>10} {'Sig':>5}")
print("-" * 75)
for m in models:
    vals = scores[m]
    mean, std = np.mean(vals), np.std(vals)
    diff = mean - np.mean(text_scores)
    if m == "text_only":
        print(f"{m:<22} {mean:>9.4f} {std:>7.4f} {'—':>14} {'—':>10} {'—':>5}")
    else:
        t, p = stats.ttest_rel(vals, text_scores)
        sig = "***" if p<0.001 else "**" if p<0.01 else "*" if p<0.05 else "n.s."
        print(f"{m:<22} {mean:>9.4f} {std:>7.4f} {diff:>+14.4f} {p:>10.4f} {sig:>5}")
print("=" * 75)
print("Significance: * p<0.05  ** p<0.01  *** p<0.001  n.s. = not significant")


Model                    Mean F1     Std   vs text-only    p-value   Sig
---------------------------------------------------------------------------
text_only                 0.8119  0.0175              —          —     —
image_only_r50            0.6294  0.0061        -0.1824     0.0001   ***
image_only_r18            0.6345  0.0203        -0.1774     0.0001   ***
late_fusion_r50           0.8126  0.0242        +0.0007     0.9119  n.s.
late_fusion_r18           0.8290  0.0152        +0.0171     0.0808  n.s.
gated_fusion              0.8082  0.0310        -0.0037     0.8388  n.s.
Significance: * p<0.05  ** p<0.01  *** p<0.001  n.s. = not significant


---
## Part 4 — Error Analysis & GradCAM Visualization

### Step C1: Find misclassified samples (late fusion R50)

In [20]:
import torch, numpy as np, pandas as pd
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
LABEL_NAMES = ["fashion_beauty","food_travel","knowledge_tutorial"]

class LateFusion50(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(2816,512),nn.ReLU(),nn.Dropout(0.3),nn.Linear(512,3))
    def forward(self,t,v): return self.net(torch.cat([t,v],dim=1))

txt_feat = np.load("features/text_features.npy")
txt_lbl  = np.load("features/text_labels.npy")
txt_ids  = np.load("features/text_ids.npy")
img50    = np.load("features/image_features.npy")
meta     = pd.read_csv("data/metadata_balanced.csv")

test_ids = set(pd.read_csv("data/splits/test.csv")["note_idx"].tolist())
mask     = np.array([i in test_ids for i in txt_ids])
Xt = torch.tensor(txt_feat[mask], dtype=torch.float32)
Xi = torch.tensor(img50[mask],    dtype=torch.float32)
y  = torch.tensor(txt_lbl[mask],  dtype=torch.long)
test_note_ids = txt_ids[mask]

model = LateFusion50().to(DEVICE)
model.load_state_dict(torch.load("results/late_fusion_best.pt", map_location=DEVICE))
model.eval()

with torch.no_grad():
    logits = model(Xt.to(DEVICE), Xi.to(DEVICE))
preds = logits.argmax(1).cpu().numpy()

# Find errors
errors = []
for i, (pred, true, nid) in enumerate(zip(preds, y.numpy(), test_note_ids)):
    if pred != true:
        row = meta[meta["note_idx"]==nid].iloc[0]
        errors.append({
            "note_idx": nid,
            "true": LABEL_NAMES[true],
            "pred": LABEL_NAMES[pred],
            "title": row["title"],
            "content": str(row["content"])[:100],
            "image_path": row["image_path"],
            "content_length": row["content_length"],
        })

print(f"Total test samples : 72")
print(f"Misclassified      : {len(errors)}")
print(f"Accuracy           : {(72-len(errors))/72:.4f}")
print()
error_df = pd.DataFrame(errors)
print(error_df[["true","pred","content_length","title"]].to_string())
error_df.to_csv("results/error_cases.csv", index=False)
print("\nSaved to results/error_cases.csv")


Total test samples : 72
Misclassified      : 12
Accuracy           : 0.8333

                  true                pred  content_length                     title
0       fashion_beauty  knowledge_tutorial              85                🐟美妆来自火星的软妹
1          food_travel  knowledge_tutorial             240           大早上起来看到这一幕，吓我一跳
2   knowledge_tutorial         food_travel             198  油焖虾本身并不能直接减脂。\n \n油焖虾一般需
3          food_travel      fashion_beauty              63     沪上阿姨 | 羽衣甘蓝纤体瓶 不是鲜榨的？
4       fashion_beauty  knowledge_tutorial             146         家里真的有必要备一台立式挂烫机吗？
5       fashion_beauty         food_travel              89                这玩意要148过分吗
6   knowledge_tutorial      fashion_beauty              72                 大白罐真的很好用！
7          food_travel  knowledge_tutorial             207               九阳破壁机测评有人看吗
8       fashion_beauty  knowledge_tutorial             738     70元的华为半入耳式耳机，超长续航40小时
9          food_travel  knowledge_tutorial              35        福田口岸的自助

### Step C2: Error pattern analysis

In [21]:
import pandas as pd, numpy as np, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

error_df = pd.read_csv("results/error_cases.csv")
LABEL_NAMES = ["fashion_beauty","food_travel","knowledge_tutorial"]

# Confusion pairs
print("Most common error patterns:")
pairs = error_df.groupby(["true","pred"]).size().reset_index(name="count").sort_values("count",ascending=False)
print(pairs.to_string(index=False))

# Error rate by text length bucket
bins = [0, 201, 521, 9999]
labels = ["Short (<=201)", "Medium (202-521)", "Long (>521)"]
error_df["length_bucket"] = pd.cut(error_df["content_length"], bins=bins, labels=labels)

# All test samples length bucket
meta = pd.read_csv("data/metadata_balanced.csv")
test_ids = set(pd.read_csv("data/splits/test.csv")["note_idx"].tolist())
test_df  = meta[meta["note_idx"].isin(test_ids)].copy()
test_df["length_bucket"] = pd.cut(test_df["content_length"], bins=bins, labels=labels)

total_per_bucket = test_df["length_bucket"].value_counts()
error_per_bucket = error_df["length_bucket"].value_counts()
error_rate = (error_per_bucket / total_per_bucket).fillna(0)

print("\nError rate by text length:")
for b in labels:
    n_err = error_per_bucket.get(b, 0)
    n_tot = total_per_bucket.get(b, 0)
    print(f"  {b}: {n_err}/{n_tot} = {n_err/n_tot:.1%}" if n_tot>0 else f"  {b}: 0/0")

# Plot error rate by bucket
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
error_rate.reindex(labels).plot(kind="bar", ax=axes[0], color=["#E74C3C","#3498DB","#2ECC71"])
axes[0].set_title("Error Rate by Text Length")
axes[0].set_ylabel("Error Rate"); axes[0].set_xticklabels(labels, rotation=15)
axes[0].axhline(len(error_df)/72, color="gray", linestyle="--", label="Overall error rate")
axes[0].legend()

# Error count by true class
error_df["true"].value_counts().reindex(LABEL_NAMES).plot(kind="bar", ax=axes[1],
    color=["#E74C3C","#3498DB","#2ECC71"])
axes[1].set_title("Error Count by True Class")
axes[1].set_ylabel("Number of errors"); axes[1].set_xticklabels(LABEL_NAMES, rotation=15)
plt.tight_layout()
plt.savefig("results/figures/07_error_analysis.png", dpi=150, bbox_inches="tight")
plt.close()
print("\nSaved: results/figures/07_error_analysis.png")


Most common error patterns:
              true               pred  count
    fashion_beauty knowledge_tutorial      4
       food_travel knowledge_tutorial      3
knowledge_tutorial     fashion_beauty      2
    fashion_beauty        food_travel      1
       food_travel     fashion_beauty      1
knowledge_tutorial        food_travel      1

Error rate by text length:
  Short (<=201): 8/24 = 33.3%
  Medium (202-521): 2/23 = 8.7%
  Long (>521): 2/25 = 8.0%

Saved: results/figures/07_error_analysis.png


### Step C3: GradCAM on misclassified images

In [22]:
!pip install grad-cam -q


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 107.4 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [28]:
import torch, torch.nn as nn, numpy as np, pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from PIL import Image
from torchvision.models import resnet50, ResNet50_Weights
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
import os
import matplotlib.pyplot as plt
from matplotlib import font_manager
import subprocess

subprocess.run(["apt-get", "install", "-y", "fonts-noto-cjk"],
               capture_output=True)
font_manager.fontManager.addfont("/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc")
plt.rcParams["font.family"] = "Noto Sans CJK JP"
plt.rcParams["axes.unicode_minus"] = False


DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
LABEL_NAMES = ["fashion_beauty","food_travel","knowledge_tutorial"]
IMG_ROOT = "data/images"

weights   = ResNet50_Weights.DEFAULT
cam_model = resnet50(weights=weights)
cam_model.fc = nn.Sequential(nn.Linear(2048,256),nn.ReLU(),nn.Dropout(0.2),nn.Linear(256,3))
cam_model.load_state_dict(torch.load("results/image_only_best.pt", map_location=DEVICE), strict=False)
cam_model = cam_model.to(DEVICE).eval()

transform = weights.transforms()
error_df  = pd.read_csv("results/error_cases.csv").head(6)

target_layer = [cam_model.layer4[-1]]
cam = GradCAM(model=cam_model, target_layers=target_layer)

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for idx, (_, row) in enumerate(error_df.iterrows()):
    rel  = row["image_path"].replace("image/", "", 1)
    path = os.path.join(IMG_ROOT, rel)
    try:
        pil_img = Image.open(path).convert("RGB").resize((224, 224))
        img_arr = np.array(pil_img, dtype=np.float32) / 255.0
        input_t = transform(Image.open(path).convert("RGB")).unsqueeze(0).to(DEVICE)

        target_cat    = LABEL_NAMES.index(row["pred"])
        grayscale_cam = cam(input_tensor=input_t,
                            targets=[ClassifierOutputTarget(target_cat)])[0]
        viz = show_cam_on_image(img_arr, grayscale_cam, use_rgb=True)

        axes[idx].imshow(viz)
        title_str = f"True: {row['true']}\nPred: {row['pred']}\n\"{str(row['title'])[:30]}...\""
        axes[idx].set_title(title_str, fontsize=8)
        axes[idx].axis("off")
    except Exception as e:
        axes[idx].text(0.5, 0.5, f"Image not available\n{e}", ha="center", va="center")
        axes[idx].axis("off")

plt.suptitle("GradCAM: What the model looks at in misclassified images", fontsize=13)
plt.tight_layout()
plt.savefig("results/figures/08_gradcam_errors.png", dpi=150, bbox_inches="tight")
plt.close()
print("Saved: results/figures/08_gradcam_errors.png")

/tmp/ipykernel_2376/3614708654.py:63: UserWarning: Glyph 128031 (\N{FISH}) missing from font(s) Noto Sans CJK JP.
  plt.tight_layout()
/tmp/ipykernel_2376/3614708654.py:64: UserWarning: Glyph 128031 (\N{FISH}) missing from font(s) Noto Sans CJK JP.
  plt.savefig("results/figures/08_gradcam_errors.png", dpi=150, bbox_inches="tight")


Saved: results/figures/08_gradcam_errors.png


### Step C4: Download all new figures

In [29]:
from google.colab import files
import shutil

shutil.make_archive("all_figures", "zip", "results/figures")
files.download("all_figures.zip")
np.save("results/multiseed_results.npy", all_results)
print("Done — download all_figures.zip")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Done — download all_figures.zip
